# 01F — Clean Leakage-Safe Feature Engineering with Official Auxiliary Features

This notebook updates the previous official-auxiliary feature engineering pipeline to follow the cleaned markdown plan.

Main changes:

- keep only reproducible future-safe features
- remove weak/stale broad all-history features
- remove gender-share features
- remove broad month-level promo lift features
- use recent3y / recent-vs-all / trend features for changing business signals
- avoid raw payment sums and sum-installment features
- avoid actual or previous-month future auxiliary activity

Revenue/COGS lag features are still recursively updated during modeling. Auxiliary variables are **not** forecasted alongside Revenue/COGS.


In [10]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import json
import numpy as np
import pandas as pd
import os
import math
import re

os.chdir("..")
Path = "./dataset"

RANDOM_STATE = 42

TRAIN_END_FINAL = pd.Timestamp("2022-12-31")
BACKTEST_TRAIN_END = pd.Timestamp("2020-12-31")
BACKTEST_START = pd.Timestamp("2021-01-01")
BACKTEST_END = pd.Timestamp("2022-12-31")
FORECAST_START = pd.Timestamp("2023-01-01")
FORECAST_END = pd.Timestamp("2024-07-01")

TARGETS = ["Revenue", "COGS"]

def safe_divide(a, b):
    a_arr = pd.Series(a).astype(float).to_numpy()
    b_ser = pd.Series(b)
    b_arr = b_ser.astype(float).to_numpy()
    mask = b_ser.notna().to_numpy() & (b_arr != 0)
    return pd.Series(np.where(mask, a_arr / b_arr, np.nan))

df = pd.read_csv(Path + "/sales.csv", parse_dates=["Date"])
df = df[["Date", "Revenue", "COGS"]].copy()
df = df.sort_values("Date").reset_index(drop=True)

orders_df = pd.read_csv(Path + "/orders.csv", parse_dates=["order_date"])
customers_df = pd.read_csv(Path + "/customers.csv", parse_dates=["signup_date"])
payments_df = pd.read_csv(Path + "/payments.csv")
promotions_df = pd.read_csv(Path + "/promotions.csv",  parse_dates=["start_date", "end_date"])
inventory_df = pd.read_csv(Path + "/inventory.csv", parse_dates=["snapshot_date"])
products_df = pd.read_csv(Path + "/products.csv")

print("Sales shape after Date/Revenue/COGS sanitization:", df.shape)
print("Columns:", df.columns.tolist())
print("Date range:", df["Date"].min(), "→", df["Date"].max())
print("Last known Revenue date:", df.loc[df["Revenue"].notna(), "Date"].max())
print("Last known COGS date:", df.loc[df["COGS"].notna(), "Date"].max())

Sales shape after Date/Revenue/COGS sanitization: (3833, 3)
Columns: ['Date', 'Revenue', 'COGS']
Date range: 2012-07-04 00:00:00 → 2022-12-31 00:00:00
Last known Revenue date: 2022-12-31 00:00:00
Last known COGS date: 2022-12-31 00:00:00


## 1. Sales/date features

In [11]:
# ============================================================
# 1. Static known-in-advance features from Date only
# ============================================================

def add_static_features(df):
    """Known-ahead calendar features using only Date + target columns."""
    out = df[["Date", "Revenue", "COGS"]].copy()
    d = out["Date"]
    tau = 2 * np.pi

    # Basic calendar position
    out["year"] = d.dt.year
    out["month"] = d.dt.month
    out["day"] = d.dt.day
    out["dayofweek"] = d.dt.dayofweek
    out["dayofyear"] = d.dt.dayofyear
    out["weekofyear"] = d.dt.isocalendar().week.astype(int)
    out["quarter"] = d.dt.quarter
    out["is_weekend"] = (out["dayofweek"] >= 5).astype(int)
    out["is_month_start_calc"] = d.dt.is_month_start.astype(int)
    out["is_month_end_calc"] = d.dt.is_month_end.astype(int)
    out["is_quarter_start"] = d.dt.is_quarter_start.astype(int)
    out["is_quarter_end"] = d.dt.is_quarter_end.astype(int)

    # Month position features: allowed because they come only from Date.
    out["days_in_month"] = d.dt.days_in_month
    out["days_to_month_end"] = out["days_in_month"] - out["day"]
    out["days_since_month_start"] = out["day"] - 1
    out["month_progress"] = out["days_since_month_start"] / out["days_in_month"].clip(lower=1)
    out["week_of_month"] = ((out["day"] - 1) // 7 + 1).astype(int)
    out["is_first_1_day"] = (out["days_since_month_start"] <= 0).astype(int)
    out["is_first_3_days"] = (out["days_since_month_start"] <= 2).astype(int)
    out["is_last_1_day"] = (out["days_to_month_end"] <= 0).astype(int)
    out["is_last_3_days"] = (out["days_to_month_end"] <= 2).astype(int)
    out["is_first_week_of_month"] = (out["week_of_month"] == 1).astype(int)
    out["is_last_week_of_month"] = (out["days_to_month_end"] < 7).astype(int)
    out["is_mid_month"] = out["day"].between(10, 20).astype(int)
    out["days_to_year_end"] = (pd.to_datetime(out["year"].astype(str) + "-12-31") - d).dt.days
    out["years_since_start"] = (out["year"] - out["year"].min()) + (out["dayofyear"] / 366.0)

    # Cyclical encodings
    for k in [1, 2, 3, 4, 5]:
        out[f"sin_y{k}"] = np.sin(tau * k * out["dayofyear"] / 365.25)
        out[f"cos_y{k}"] = np.cos(tau * k * out["dayofyear"] / 365.25)
    for k in [1, 2]:
        out[f"sin_w{k}"] = np.sin(tau * k * out["dayofweek"] / 7.0)
        out[f"cos_w{k}"] = np.cos(tau * k * out["dayofweek"] / 7.0)
        out[f"sin_m{k}"] = np.sin(tau * k * out["days_since_month_start"] / out["days_in_month"].clip(lower=1))
        out[f"cos_m{k}"] = np.cos(tau * k * out["days_since_month_start"] / out["days_in_month"].clip(lower=1))

    # Regime/time-position features. These are deterministic from Date.
    out["is_odd_year"] = (out["year"] % 2 == 1).astype(int)
    out["regime_pre2019"] = (out["year"] <= 2018).astype(int)
    out["regime_2019"] = (out["year"] == 2019).astype(int)
    out["regime_post2020"] = (out["year"] >= 2020).astype(int)

    # Business-season hypotheses derived from the sales series shape, but computable from Date only.
    out["flag_q3_odd_margin_risk"] = ((out["quarter"] == 3) & (out["is_odd_year"] == 1)).astype(int)
    out["flag_aug_odd"] = ((out["month"] == 8) & (out["is_odd_year"] == 1)).astype(int)
    out["flag_q4_peak"] = (out["quarter"] == 4).astype(int)
    out["flag_q2_peak"] = (out["quarter"] == 2).astype(int)
    out["flag_low_season_q1_q3"] = out["month"].isin([1, 2, 3, 7, 8]).astype(int)

    return out


## 2. Sales target priors

In [12]:
# ============================================================
# 2. Target priors learned only from the selected training window
# ============================================================

def _target_stats_by_group(hist, group_col, target):
    g = hist.groupby(group_col)[target]
    return pd.DataFrame({
        "median": g.median(),
        "mean": g.mean(),
        "q25": g.quantile(0.25),
        "q75": g.quantile(0.75),
        "std": g.std(),
        "count": g.count(),
    })

def build_target_priors(train_hist, recent_days=730):
    """Build full-history and recent-history target priors using only training rows."""
    hist = train_hist.dropna(subset=["Revenue", "COGS"]).copy()
    hist["dayofyear"] = hist["Date"].dt.dayofyear
    hist["month"] = hist["Date"].dt.month
    hist["dayofweek"] = hist["Date"].dt.dayofweek
    hist["weekofyear"] = hist["Date"].dt.isocalendar().week.astype(int)

    hist["gross_profit"] = hist["Revenue"] - hist["COGS"]
    hist["cogs_ratio"] = np.where(hist["Revenue"] != 0, hist["COGS"] / hist["Revenue"], np.nan)
    hist["gpm"] = np.where(hist["Revenue"] != 0, (hist["Revenue"] - hist["COGS"]) / hist["Revenue"], np.nan)

    # Historical spike/drop labels built only inside training period.
    for target in TARGETS:
        lower = target.lower()
        med28 = hist[target].shift(1).rolling(28, min_periods=7).median()
        std28 = hist[target].shift(1).rolling(28, min_periods=7).std()
        diff1 = hist[target].diff()
        hist[f"{lower}_spike_event"] = (hist[target] > (med28 + 1.25*std28)).astype(int)
        hist[f"{lower}_drop_event"] = (diff1 < (-0.75*std28)).astype(int)

    priors = {}
    priors["global"] = {
        "revenue": hist["Revenue"].median(),
        "cogs": hist["COGS"].median(),
        "gross_profit": hist["gross_profit"].median(),
        "cogs_ratio": hist["cogs_ratio"].median(),
        "gpm": hist["gpm"].median(),
        "revenue_std": hist["Revenue"].std(),
        "cogs_std": hist["COGS"].std(),
        "revenue_q75": hist["Revenue"].quantile(0.75),
        "cogs_q75": hist["COGS"].quantile(0.75),
        "revenue_spike_prob": hist["revenue_spike_event"].mean(),
        "cogs_spike_prob": hist["cogs_spike_event"].mean(),
    }

    for key, group_col in [("doy", "dayofyear"), ("month", "month"), ("dow", "dayofweek"), ("week", "weekofyear")]:
        priors[f"rev_by_{key}"] = _target_stats_by_group(hist, group_col, "Revenue")
        priors[f"cogs_by_{key}"] = _target_stats_by_group(hist, group_col, "COGS")
        priors[f"gp_by_{key}"] = _target_stats_by_group(hist, group_col, "gross_profit")
        priors[f"ratio_by_{key}"] = _target_stats_by_group(hist, group_col, "cogs_ratio")
        priors[f"gpm_by_{key}"] = _target_stats_by_group(hist, group_col, "gpm")
        priors[f"rev_spike_prob_by_{key}"] = hist.groupby(group_col)["revenue_spike_event"].mean()
        priors[f"cogs_spike_prob_by_{key}"] = hist.groupby(group_col)["cogs_spike_event"].mean()

    cutoff = hist["Date"].max() - pd.Timedelta(days=recent_days)
    recent = hist.loc[hist["Date"] >= cutoff].copy()
    if len(recent) < 180:
        recent = hist.copy()
    priors["recent"] = {}
    priors["recent"]["global"] = {
        "revenue": recent["Revenue"].median(),
        "cogs": recent["COGS"].median(),
        "gross_profit": recent["gross_profit"].median(),
        "cogs_ratio": recent["cogs_ratio"].median(),
        "gpm": recent["gpm"].median(),
        "revenue_std": recent["Revenue"].std(),
        "cogs_std": recent["COGS"].std(),
    }
    for key, group_col in [("doy", "dayofyear"), ("month", "month"), ("dow", "dayofweek"), ("week", "weekofyear")]:
        priors["recent"][f"rev_by_{key}"] = _target_stats_by_group(recent, group_col, "Revenue")
        priors["recent"][f"cogs_by_{key}"] = _target_stats_by_group(recent, group_col, "COGS")
        priors["recent"][f"ratio_by_{key}"] = _target_stats_by_group(recent, group_col, "cogs_ratio")
        priors["recent"][f"gpm_by_{key}"] = _target_stats_by_group(recent, group_col, "gpm")

    priors["recent_start"] = recent["Date"].min()
    priors["recent_end"] = recent["Date"].max()
    return priors

def _map_stats(dates, priors, prefix, stat="median", global_value=None):
    doy = dates.dt.dayofyear
    mon = dates.dt.month
    dow = dates.dt.dayofweek
    wk = dates.dt.isocalendar().week.astype(int)
    s = (
        doy.map(priors[f"{prefix}_by_doy"][stat])
        .fillna(wk.map(priors[f"{prefix}_by_week"][stat]))
        .fillna(mon.map(priors[f"{prefix}_by_month"][stat]))
        .fillna(dow.map(priors[f"{prefix}_by_dow"][stat]))
    )
    if global_value is not None:
        s = s.fillna(global_value)
    return s

def _map_prob(dates, priors, prefix, global_value=0.0):
    doy = dates.dt.dayofyear
    mon = dates.dt.month
    dow = dates.dt.dayofweek
    wk = dates.dt.isocalendar().week.astype(int)
    return (
        doy.map(priors[f"{prefix}_by_doy"])
        .fillna(wk.map(priors[f"{prefix}_by_week"]))
        .fillna(mon.map(priors[f"{prefix}_by_month"]))
        .fillna(dow.map(priors[f"{prefix}_by_dow"]))
        .fillna(global_value)
    )

def attach_target_priors(df, priors):
    out = df.copy()
    dates = out["Date"]
    mon = dates.dt.month

    g = priors["global"]
    rg = priors["recent"]["global"]

    # Full-history target priors
    out["prior_revenue_blend"] = _map_stats(dates, priors, "rev", "median", g["revenue"])
    out["prior_cogs_blend"] = _map_stats(dates, priors, "cogs", "median", g["cogs"])
    out["prior_gp_blend"] = _map_stats(dates, priors, "gp", "median", g["gross_profit"])
    out["prior_cogs_ratio_blend"] = _map_stats(dates, priors, "ratio", "median", g["cogs_ratio"])
    out["prior_gpm_blend"] = _map_stats(dates, priors, "gpm", "median", g["gpm"])

    # More granular full-history priors
    for stat in ["mean", "q25", "q75", "std"]:
        out[f"prior_revenue_{stat}"] = _map_stats(dates, priors, "rev", stat, g.get(f"revenue_{stat}", g["revenue"]))
        out[f"prior_cogs_{stat}"] = _map_stats(dates, priors, "cogs", stat, g.get(f"cogs_{stat}", g["cogs"]))

    out["prior_revenue_iqr"] = out["prior_revenue_q75"] - out["prior_revenue_q25"]
    out["prior_cogs_iqr"] = out["prior_cogs_q75"] - out["prior_cogs_q25"]
    out["prior_revenue_cv"] = safe_divide(out["prior_revenue_std"], out["prior_revenue_blend"])
    out["prior_cogs_cv"] = safe_divide(out["prior_cogs_std"], out["prior_cogs_blend"])

    # Month priors explicitly preserved for models that like coarser seasonal anchors
    out["prior_revenue_month"] = mon.map(priors["rev_by_month"]["median"]).fillna(g["revenue"])
    out["prior_cogs_month"] = mon.map(priors["cogs_by_month"]["median"]).fillna(g["cogs"])
    out["prior_cogs_ratio_month"] = mon.map(priors["ratio_by_month"]["median"]).fillna(g["cogs_ratio"])
    out["prior_gpm_month"] = mon.map(priors["gpm_by_month"]["median"]).fillna(g["gpm"])

    # Recent 2-year target priors, still train-window-only
    rp = priors["recent"]
    out["prior_revenue_blend_recent2y"] = _map_stats(dates, rp, "rev", "median", rg["revenue"])
    out["prior_cogs_blend_recent2y"] = _map_stats(dates, rp, "cogs", "median", rg["cogs"])
    out["prior_cogs_ratio_blend_recent2y"] = _map_stats(dates, rp, "ratio", "median", rg["cogs_ratio"])
    out["prior_gpm_blend_recent2y"] = _map_stats(dates, rp, "gpm", "median", rg["gpm"])

    for stat in ["q25", "q75", "std"]:
        out[f"prior_revenue_{stat}_recent2y"] = _map_stats(dates, rp, "rev", stat, rg["revenue"])
        out[f"prior_cogs_{stat}_recent2y"] = _map_stats(dates, rp, "cogs", stat, rg["cogs"])
    out["prior_revenue_iqr_recent2y"] = out["prior_revenue_q75_recent2y"] - out["prior_revenue_q25_recent2y"]
    out["prior_cogs_iqr_recent2y"] = out["prior_cogs_q75_recent2y"] - out["prior_cogs_q25_recent2y"]

    # Recent vs all drift regime
    out["prior_revenue_recent_vs_all"] = safe_divide(out["prior_revenue_blend_recent2y"], out["prior_revenue_blend"])
    out["prior_cogs_recent_vs_all"] = safe_divide(out["prior_cogs_blend_recent2y"], out["prior_cogs_blend"])
    out["prior_ratio_recent_minus_all"] = out["prior_cogs_ratio_blend_recent2y"] - out["prior_cogs_ratio_blend"]

    # Historical spike probability by date regime
    out["prior_revenue_spike_prob"] = _map_prob(dates, priors, "rev_spike_prob", g["revenue_spike_prob"])
    out["prior_cogs_spike_prob"] = _map_prob(dates, priors, "cogs_spike_prob", g["cogs_spike_prob"])

    # Regime buckets based only on train-window priors. These are safe for future dates.
    out["date_regime_revenue_level"] = pd.qcut(out["prior_revenue_blend"].rank(method="first"), 5, labels=False, duplicates="drop").astype(float)
    out["date_regime_cogs_level"] = pd.qcut(out["prior_cogs_blend"].rank(method="first"), 5, labels=False, duplicates="drop").astype(float)
    out["date_regime_volatility"] = pd.qcut(out["prior_revenue_cv"].fillna(out["prior_revenue_cv"].median()).rank(method="first"), 4, labels=False, duplicates="drop").astype(float)
    out["date_regime_margin"] = pd.qcut(out["prior_gpm_blend"].rank(method="first"), 4, labels=False, duplicates="drop").astype(float)


    return out


## 3. Official auxiliary clean features

This section converts official auxiliary files into **clean, no-leakage features**:

- orders/payments → order pressure, AOV, payment-method shares, installment shares
- customers/orders → customer lifecycle expectations only
- products/inventory → product-category mix expectations
- inventory → recent inventory-pressure rates only

Removed/deprioritized by design:

- gender-share features
- broad all-history promo lift averages by month
- raw payment-value sums
- sum of installments
- broad all-history inventory pressure averages
- actual or previous-month auxiliary activity


In [13]:
# ============================================================
# 3. Official auxiliary clean historical-expectation features
# ============================================================
# This version follows the clean feature plan:
# - No actual future auxiliary activity.
# - No previous forecast-month auxiliary values.
# - No broad all-history averages for changing business actions when a recent/trend version is better.
# - No gender-share features.
# - No broad month-level promo lift averages.
# - No raw payment-value sums or sum-installment features.

def _add_calendar_keys(x, date_col="Date"):
    out = x.copy()
    d = pd.to_datetime(out[date_col])
    out["monthofyear"] = d.dt.month.astype(int)
    out["quarter_key"] = d.dt.quarter.astype(int)
    out["weekofyear_key"] = d.dt.isocalendar().week.astype(int)
    out["dayofweek_key"] = d.dt.dayofweek.astype(int)
    out["dayofmonth_key"] = d.dt.day.astype(int)
    out["month_dayofweek_key"] = out["monthofyear"].astype(str) + "_" + out["dayofweek_key"].astype(str)
    out["year_key"] = d.dt.year.astype(int)
    return out

def _clean_token(x):
    x = str(x).lower()
    x = re.sub(r"\b20\d{2}\b", "", x)
    x = re.sub(r"[^a-z0-9]+", "_", x).strip("_")
    return x or "unknown"

def _entropy_from_shares(frame, cols):
    arr = frame[cols].fillna(0).clip(lower=0).to_numpy(dtype=float)
    row_sum = arr.sum(axis=1, keepdims=True)
    p = np.divide(arr, row_sum, out=np.zeros_like(arr), where=row_sum != 0)
    with np.errstate(divide="ignore", invalid="ignore"):
        ent = -np.nansum(np.where(p > 0, p * np.log(p), 0.0), axis=1)
    return ent

def _merge_group_prior(feat, table, key_col, source_cols):
    if table is None or table.empty:
        return feat
    use_cols = [key_col] + [c for c in source_cols if c in table.columns]
    return feat.merge(table[use_cols].drop_duplicates(key_col), on=key_col, how="left")

def _recent_start_for_aux(train_end, years=3):
    train_end = pd.Timestamp(train_end)
    return max(train_end - pd.DateOffset(years=years) + pd.Timedelta(days=1), pd.Timestamp("1900-01-01"))

def _mean_by_key(frame, key, cols, rename_prefix, rename_suffix):
    if frame is None or frame.empty:
        return None
    cols = [c for c in cols if c in frame.columns]
    if not cols:
        return None
    tab = frame.groupby(key)[cols].mean().reset_index()
    rename = {c: f"{rename_prefix}{c.replace('aux_', '')}_{rename_suffix}" for c in cols}
    return tab.rename(columns=rename)

def _recent_vs_all(out, base_col, recent_col, new_col):
    if base_col in out.columns and recent_col in out.columns:
        out[new_col] = safe_divide(out[recent_col], out[base_col]).values
    return out

def _trend_slope_by_month_year(daily_or_monthly, date_col, metric_col, output_name):
    """
    For each month-of-year, compute a simple slope of yearly average metric ~ year.
    Uses only rows already filtered to train_end.
    """
    if daily_or_monthly is None or daily_or_monthly.empty or metric_col not in daily_or_monthly.columns:
        return None
    x = daily_or_monthly[[date_col, metric_col]].copy()
    x[date_col] = pd.to_datetime(x[date_col], errors="coerce")
    x = x[x[date_col].notna()].copy()
    x["monthofyear"] = x[date_col].dt.month.astype(int)
    x["year_key"] = x[date_col].dt.year.astype(int)
    y = (
        x.groupby(["monthofyear", "year_key"])[metric_col]
        .mean()
        .reset_index()
        .dropna(subset=[metric_col])
    )
    rows = []
    for m, sub in y.groupby("monthofyear"):
        if sub["year_key"].nunique() < 2:
            slope = np.nan
        else:
            try:
                slope = np.polyfit(sub["year_key"].astype(float), sub[metric_col].astype(float), 1)[0]
            except Exception:
                slope = np.nan
        rows.append({"monthofyear": m, output_name: slope})
    return pd.DataFrame(rows)

def build_orders_payment_daily(orders, payments, train_end, calendar_dates):
    """
    Daily order/payment behavior using only orders <= train_end.
    Future rows receive historical expectations later; actual future order/payment values are never used.
    """
    train_end = pd.Timestamp(train_end)
    cal = pd.DataFrame({"Date": pd.date_range(pd.Timestamp(calendar_dates.min()), train_end, freq="D")})

    default_cols = dict(
        aux_order_count=0.0,
        aux_aov=np.nan,
        aux_delivered_rate=np.nan,
        aux_returned_rate=np.nan,
        aux_cancelled_rate=np.nan,
        aux_paid_rate=np.nan,
        aux_credit_payment_share=np.nan,
        aux_cod_payment_share=np.nan,
        aux_bank_payment_share=np.nan,
        aux_payment_method_entropy=np.nan,
        aux_avg_installments=np.nan,
        aux_share_installment_payment=np.nan,
        aux_share_high_installment_payment=np.nan,
        aux_high_value_payment_share=np.nan,
    )

    if orders is None or orders.empty or "order_date" not in orders.columns:
        return cal.assign(**default_cols)

    od = orders.copy()
    od["order_date"] = pd.to_datetime(od["order_date"], errors="coerce")
    od = od[(od["order_date"].notna()) & (od["order_date"] <= train_end)].copy()
    if od.empty:
        return cal.assign(**default_cols)
    od["Date"] = od["order_date"].dt.floor("D")

    if payments is not None and not payments.empty and "order_id" in payments.columns:
        pay = payments.copy()
        pay["payment_value"] = pd.to_numeric(pay.get("payment_value", np.nan), errors="coerce")
        pay["installments"] = pd.to_numeric(pay.get("installments", np.nan), errors="coerce")
        if "payment_method" in pay.columns:
            pay["payment_method_from_payment"] = pay["payment_method"]
        else:
            pay["payment_method_from_payment"] = np.nan
        pay_order = pay.groupby("order_id", as_index=False).agg(
            payment_value=("payment_value", "sum"),
            installments=("installments", "mean"),
            payment_method_from_payment=("payment_method_from_payment", "first"),
        )
        od = od.merge(pay_order, on="order_id", how="left")
    else:
        od["payment_value"] = np.nan
        od["installments"] = np.nan
        od["payment_method_from_payment"] = np.nan

    if "payment_method" in od.columns:
        od["payment_method_final"] = od["payment_method"].fillna(od["payment_method_from_payment"])
    else:
        od["payment_method_final"] = od["payment_method_from_payment"]

    status = od["order_status"].astype(str).str.lower() if "order_status" in od.columns else pd.Series("", index=od.index)
    method = od["payment_method_final"].astype(str).str.lower()

    od["is_delivered"] = status.str.contains("delivered|completed|shipped", regex=True, na=False).astype(int)
    od["is_returned"] = status.str.contains("returned|return", regex=True, na=False).astype(int)
    od["is_cancelled"] = status.str.contains("cancel|canceled|cancelled", regex=True, na=False).astype(int)
    od["is_paid"] = (~status.str.contains("cancel|returned|return", regex=True, na=False)).astype(int)

    od["is_credit"] = method.str.contains("credit", na=False).astype(int)
    od["is_cod"] = method.str.contains("cod|cash", regex=True, na=False).astype(int)
    od["is_bank"] = method.str.contains("bank|transfer", regex=True, na=False).astype(int)
    od["is_installment"] = (od["installments"].fillna(1) > 1).astype(int)
    od["is_high_installment"] = (od["installments"].fillna(1) >= 6).astype(int)

    high_value_threshold = od["payment_value"].quantile(0.80) if od["payment_value"].notna().any() else np.nan
    od["is_high_value_payment"] = (od["payment_value"] >= high_value_threshold).astype(int) if pd.notna(high_value_threshold) else 0

    daily = od.groupby("Date").agg(
        aux_order_count=("order_id", "nunique"),
        aux_payment_value_total=("payment_value", "sum"),
        aux_delivered_rate=("is_delivered", "mean"),
        aux_returned_rate=("is_returned", "mean"),
        aux_cancelled_rate=("is_cancelled", "mean"),
        aux_paid_rate=("is_paid", "mean"),
        aux_credit_payment_share=("is_credit", "mean"),
        aux_cod_payment_share=("is_cod", "mean"),
        aux_bank_payment_share=("is_bank", "mean"),
        aux_avg_installments=("installments", "mean"),
        aux_share_installment_payment=("is_installment", "mean"),
        aux_share_high_installment_payment=("is_high_installment", "mean"),
        aux_high_value_payment_share=("is_high_value_payment", "mean"),
    ).reset_index()

    daily["aux_aov"] = np.where(
        daily["aux_order_count"] > 0,
        daily["aux_payment_value_total"] / daily["aux_order_count"],
        np.nan,
    )

    out = cal.merge(daily, on="Date", how="left")
    out["aux_order_count"] = out["aux_order_count"].fillna(0)

    # Raw payment sum is only used internally to build AOV/high-value flags. It is not exported as a model feature.
    share_cols = ["aux_credit_payment_share", "aux_cod_payment_share", "aux_bank_payment_share"]
    out["aux_payment_method_entropy"] = _entropy_from_shares(out, share_cols)
    return out

def attach_orders_payment_expectations(feat, orders, payments, train_end):
    daily = build_orders_payment_daily(orders, payments, train_end, feat["Date"])
    d = _add_calendar_keys(daily, "Date")
    train_end = pd.Timestamp(train_end)
    recent = d[d["Date"] >= _recent_start_for_aux(train_end, years=3)].copy()

    out = _add_calendar_keys(feat, "Date")

    # Strong order-pressure and AOV features.
    core_cols = ["aux_order_count", "aux_aov"]
    for key, suffix in [
        ("monthofyear", "by_monthofyear"),
        ("weekofyear_key", "by_weekofyear"),
        ("dayofweek_key", "by_dayofweek"),
        ("month_dayofweek_key", "by_month_dayofweek"),
    ]:
        tab = _mean_by_key(d, key, core_cols, "hist_expected_", suffix)
        if tab is not None:
            out = _merge_group_prior(out, tab, key, [c for c in tab.columns if c != key])

    recent_tab = _mean_by_key(recent, "monthofyear", core_cols, "hist_recent3y_expected_", "by_monthofyear")
    if recent_tab is not None:
        out = _merge_group_prior(out, recent_tab, "monthofyear", [c for c in recent_tab.columns if c != "monthofyear"])

    # Recent-vs-all ratios and trend slopes.
    out = _recent_vs_all(
        out,
        "hist_expected_order_count_by_monthofyear",
        "hist_recent3y_expected_order_count_by_monthofyear",
        "hist_order_count_recent_vs_all_ratio_by_monthofyear",
    )
    out = _recent_vs_all(
        out,
        "hist_expected_aov_by_monthofyear",
        "hist_recent3y_expected_aov_by_monthofyear",
        "hist_aov_recent_vs_all_ratio_by_monthofyear",
    )

    for metric, outname in [
        ("aux_order_count", "hist_order_count_trend_slope_by_monthofyear"),
        ("aux_aov", "hist_aov_trend_slope_by_monthofyear"),
    ]:
        slope = _trend_slope_by_month_year(d, "Date", metric, outname)
        if slope is not None:
            out = _merge_group_prior(out, slope, "monthofyear", [outname])

    # Order quality and payment behavior: use recent3y expectations only.
    quality_cols = [
        "aux_delivered_rate", "aux_returned_rate", "aux_cancelled_rate", "aux_paid_rate",
        "aux_credit_payment_share", "aux_cod_payment_share", "aux_bank_payment_share",
        "aux_payment_method_entropy", "aux_avg_installments", "aux_share_installment_payment",
        "aux_share_high_installment_payment", "aux_high_value_payment_share",
    ]
    quality_tab = _mean_by_key(recent, "monthofyear", quality_cols, "hist_recent3y_", "by_monthofyear")
    if quality_tab is not None:
        out = _merge_group_prior(out, quality_tab, "monthofyear", [c for c in quality_tab.columns if c != "monthofyear"])

    return out.drop(columns=["monthofyear", "quarter_key", "weekofyear_key", "dayofweek_key", "dayofmonth_key", "month_dayofweek_key", "year_key"], errors="ignore")

def attach_customer_lifecycle_expectations(feat, orders, customers, train_end):
    """
    Customer features are limited to lifecycle expectations.
    Gender shares and age shares are excluded by default because they were judged weak/stable.
    """
    if orders is None or customers is None or orders.empty or customers.empty:
        return feat

    train_end = pd.Timestamp(train_end)
    od = orders.copy()
    od["order_date"] = pd.to_datetime(od["order_date"], errors="coerce")
    od = od[(od["order_date"].notna()) & (od["order_date"] <= train_end)].copy()
    if od.empty or "customer_id" not in od.columns:
        return feat

    od["Date"] = od["order_date"].dt.floor("D")
    od = od.sort_values(["customer_id", "Date"])
    od["prev_order_date"] = od.groupby("customer_id")["Date"].shift(1)
    od["days_since_prev_order"] = (od["Date"] - od["prev_order_date"]).dt.days
    od["is_new_customer_order"] = od["prev_order_date"].isna().astype(int)
    od["is_returning_customer_order"] = od["prev_order_date"].notna().astype(int)
    od["is_reactivated_365_order"] = (od["days_since_prev_order"] > 365).astype(int)

    daily = od.groupby("Date").agg(
        aux_new_customer_count=("is_new_customer_order", "sum"),
        aux_returning_customer_share=("is_returning_customer_order", "mean"),
        aux_reactivation_count=("is_reactivated_365_order", "sum"),
        aux_repeat_purchase_gap_median=("days_since_prev_order", "median"),
    ).reset_index()

    cal = pd.DataFrame({"Date": pd.date_range(feat["Date"].min(), train_end, freq="D")})
    daily = cal.merge(daily, on="Date", how="left")
    daily[["aux_new_customer_count", "aux_reactivation_count"]] = daily[["aux_new_customer_count", "aux_reactivation_count"]].fillna(0)

    d = _add_calendar_keys(daily, "Date")
    recent = d[d["Date"] >= _recent_start_for_aux(train_end, years=3)].copy()
    metric_cols = ["aux_new_customer_count", "aux_returning_customer_share", "aux_reactivation_count", "aux_repeat_purchase_gap_median"]

    out = _add_calendar_keys(feat, "Date")
    for key, suffix in [("monthofyear", "by_monthofyear"), ("weekofyear_key", "by_weekofyear")]:
        tab = _mean_by_key(recent, key, metric_cols, "hist_recent3y_expected_", suffix)
        if tab is not None:
            out = _merge_group_prior(out, tab, key, [c for c in tab.columns if c != key])

    return out.drop(columns=["monthofyear", "quarter_key", "weekofyear_key", "dayofweek_key", "dayofmonth_key", "month_dayofweek_key", "year_key"], errors="ignore")

def _infer_promo_schedule(promotions, train_end):
    """
    Infer recurring scheduled promotions from official historical promotions only.
    Returns a compact schedule table by promo family.
    """
    if promotions is None or promotions.empty or not {"start_date", "end_date"}.issubset(promotions.columns):
        return pd.DataFrame()

    prom = promotions.copy()
    prom["start_date"] = pd.to_datetime(prom["start_date"], errors="coerce")
    prom["end_date"] = pd.to_datetime(prom["end_date"], errors="coerce")
    prom = prom[(prom["start_date"].notna()) & (prom["end_date"].notna()) & (prom["start_date"] <= pd.Timestamp(train_end))].copy()
    if prom.empty:
        return pd.DataFrame()

    name_col = "promo_name" if "promo_name" in prom.columns else ("promo_type" if "promo_type" in prom.columns else None)
    if name_col is None:
        prom["promo_family"] = "promo"
    else:
        prom["promo_family"] = prom[name_col].map(_clean_token)

    prom["start_month"] = prom["start_date"].dt.month
    prom["start_day"] = prom["start_date"].dt.day
    prom["duration_days"] = (prom["end_date"] - prom["start_date"]).dt.days.clip(lower=0) + 1
    prom["year_key"] = prom["start_date"].dt.year
    prom["discount_value"] = pd.to_numeric(prom.get("discount_value", np.nan), errors="coerce")
    prom["min_order_value"] = pd.to_numeric(prom.get("min_order_value", np.nan), errors="coerce")

    rows = []
    for fam, sub in prom.groupby("promo_family"):
        years = sorted(sub["year_key"].dropna().astype(int).unique().tolist())
        if len(years) < 2:
            # Too little evidence to infer recurrence.
            continue
        odd_only = all(y % 2 == 1 for y in years)
        even_only = all(y % 2 == 0 for y in years)
        # Annual-ish if seen in at least 3 training years, otherwise allow odd/even patterns if repeated.
        recurrence = "annual" if len(years) >= 3 and not odd_only and not even_only else ("odd" if odd_only else ("even" if even_only else "annual"))
        rows.append({
            "promo_family": fam,
            "start_month": int(round(sub["start_month"].median())),
            "start_day": int(round(sub["start_day"].median())),
            "duration_days": int(round(sub["duration_days"].median())),
            "discount_depth": float(sub["discount_value"].median()) if sub["discount_value"].notna().any() else 0.0,
            "min_order_value": float(sub["min_order_value"].median()) if sub["min_order_value"].notna().any() else 0.0,
            "recurrence": recurrence,
            "n_years_observed": len(years),
        })
    return pd.DataFrame(rows)

def _calculate_promo_type_effects(promotions, sales_df, train_end):
    """
    Recent/regime-aware promo-type effects.
    This avoids broad all-history month-level promo lift averages.
    """
    train_end = pd.Timestamp(train_end)
    recent_start = _recent_start_for_aux(train_end, years=3)
    sched = _infer_promo_schedule(promotions, train_end)
    if sched.empty:
        return pd.DataFrame()

    hist = sales_df[(sales_df["Date"] <= train_end) & (sales_df["Date"] >= recent_start)][["Date", "Revenue", "COGS"]].copy()
    if hist.empty:
        return pd.DataFrame()
    hist = _add_calendar_keys(hist, "Date")
    hist["gpm"] = safe_divide(hist["Revenue"] - hist["COGS"], hist["Revenue"]).values

    # Mark actual historical promo days by family.
    prom = promotions.copy()
    prom["start_date"] = pd.to_datetime(prom["start_date"], errors="coerce")
    prom["end_date"] = pd.to_datetime(prom["end_date"], errors="coerce")
    prom = prom[(prom["start_date"].notna()) & (prom["end_date"].notna()) & (prom["start_date"] <= train_end) & (prom["end_date"] >= recent_start)].copy()
    name_col = "promo_name" if "promo_name" in prom.columns else ("promo_type" if "promo_type" in prom.columns else None)
    prom["promo_family"] = prom[name_col].map(_clean_token) if name_col else "promo"

    rows = []
    for fam, subprom in prom.groupby("promo_family"):
        mask = pd.Series(False, index=hist.index)
        for _, r in subprom.iterrows():
            start = max(r["start_date"], recent_start)
            end = min(r["end_date"], train_end)
            mask |= (hist["Date"] >= start) & (hist["Date"] <= end)
        promo_days = hist[mask].copy()
        if promo_days.empty:
            continue

        # Local baseline: same month + day-of-week non-promo days in recent regime.
        baseline_values = []
        baseline_gpm = []
        for _, rr in promo_days.iterrows():
            local = hist[
                (~mask)
                & (hist["monthofyear"] == rr["monthofyear"])
                & (hist["dayofweek_key"] == rr["dayofweek_key"])
            ]
            if local.empty:
                local = hist[(~mask) & (hist["monthofyear"] == rr["monthofyear"])]
            baseline_values.append({
                "Revenue": local["Revenue"].mean(),
                "COGS": local["COGS"].mean(),
            })
            baseline_gpm.append(local["gpm"].mean())
        base = pd.DataFrame(baseline_values)
        rows.append({
            "promo_family": fam,
            "hist_recent3y_promo_lift_ratio_revenue_by_promo_type": safe_divide(promo_days["Revenue"].reset_index(drop=True), base["Revenue"]).mean(),
            "hist_recent3y_promo_lift_ratio_cogs_by_promo_type": safe_divide(promo_days["COGS"].reset_index(drop=True), base["COGS"]).mean(),
            "hist_recent3y_promo_margin_impact_by_promo_type": promo_days["gpm"].mean() - pd.Series(baseline_gpm).mean(),
            "hist_recent3y_promo_negative_margin_probability_by_promo_type": (promo_days["gpm"] < 0).mean(),
        })
    return pd.DataFrame(rows)

def attach_product_mix_expectations(feat, inventory, products, train_end):
    """
    Product mix expectations from official inventory/product data.
    Uses units_sold by category when available; otherwise skips.
    Recent3y and trend-aware versions are preferred.
    """
    if inventory is None or inventory.empty or "snapshot_date" not in inventory.columns:
        return feat

    inv = inventory.copy()
    inv["snapshot_date"] = pd.to_datetime(inv["snapshot_date"], errors="coerce")
    inv = inv[(inv["snapshot_date"].notna()) & (inv["snapshot_date"] <= pd.Timestamp(train_end))].copy()
    if inv.empty:
        return feat

    if "category" not in inv.columns and products is not None and "product_id" in inv.columns and "product_id" in products.columns and "category" in products.columns:
        inv = inv.merge(products[["product_id", "category"]], on="product_id", how="left")
    if "category" not in inv.columns or "units_sold" not in inv.columns:
        return feat

    inv["units_sold"] = pd.to_numeric(inv["units_sold"], errors="coerce").fillna(0)
    inv["Date"] = inv["snapshot_date"].dt.floor("D")
    inv = _add_calendar_keys(inv, "Date")
    inv["category_clean"] = inv["category"].map(_clean_token)

    def category_share_table(frame, prefix):
        if frame.empty:
            return None
        g = frame.groupby(["monthofyear", "category_clean"])["units_sold"].sum().reset_index()
        total = g.groupby("monthofyear")["units_sold"].transform("sum")
        g["share"] = np.where(total > 0, g["units_sold"] / total, np.nan)
        wide = g.pivot(index="monthofyear", columns="category_clean", values="share").reset_index()
        rename = {c: f"{prefix}category_share_{c}_by_monthofyear" for c in wide.columns if c != "monthofyear"}
        wide = wide.rename(columns=rename)
        share_cols = [c for c in wide.columns if c != "monthofyear"]
        if share_cols:
            wide[f"{prefix}category_mix_entropy_by_monthofyear"] = _entropy_from_shares(wide, share_cols)
        # Useful business spread if categories exist.
        street = f"{prefix}category_share_streetwear_by_monthofyear"
        outdoor = f"{prefix}category_share_outdoor_by_monthofyear"
        if street in wide.columns and outdoor in wide.columns:
            wide[f"{prefix}streetwear_minus_outdoor_share_by_monthofyear"] = wide[street] - wide[outdoor]
        return wide

    recent = inv[inv["Date"] >= _recent_start_for_aux(train_end, years=3)].copy()
    out = _add_calendar_keys(feat, "Date")

    all_tab = category_share_table(inv, "hist_expected_")
    recent_tab = category_share_table(recent, "hist_recent3y_")

    if all_tab is not None:
        out = _merge_group_prior(out, all_tab, "monthofyear", [c for c in all_tab.columns if c != "monthofyear"])
    if recent_tab is not None:
        out = _merge_group_prior(out, recent_tab, "monthofyear", [c for c in recent_tab.columns if c != "monthofyear"])

    # Recent-vs-all ratio for key category share columns.
    for c in list(out.columns):
        if c.startswith("hist_recent3y_category_share_") and c.endswith("_by_monthofyear"):
            base = c.replace("hist_recent3y_", "hist_expected_")
            if base in out.columns:
                name = c.replace("hist_recent3y_category_share_", "hist_category_share_recent_vs_all_ratio_")
                out[name] = safe_divide(out[c], out[base]).values

    # Trend slopes by month/category.
    yearly = inv.groupby(["monthofyear", "year_key", "category_clean"])["units_sold"].sum().reset_index()
    yearly_total = yearly.groupby(["monthofyear", "year_key"])["units_sold"].transform("sum")
    yearly["share"] = np.where(yearly_total > 0, yearly["units_sold"] / yearly_total, np.nan)
    rows = []
    for (m, cat), sub in yearly.dropna(subset=["share"]).groupby(["monthofyear", "category_clean"]):
        if sub["year_key"].nunique() < 2:
            slope = np.nan
        else:
            slope = np.polyfit(sub["year_key"].astype(float), sub["share"].astype(float), 1)[0]
        rows.append({"monthofyear": m, f"hist_category_share_trend_slope_{cat}_by_monthofyear": slope})
    if rows:
        trend = pd.DataFrame(rows)
        # Multiple columns from row dictionaries; group first to one row per month.
        trend = trend.groupby("monthofyear").first().reset_index()
        out = _merge_group_prior(out, trend, "monthofyear", [c for c in trend.columns if c != "monthofyear"])

    return out.drop(columns=["monthofyear", "quarter_key", "weekofyear_key", "dayofweek_key", "dayofmonth_key", "month_dayofweek_key", "year_key"], errors="ignore")

def attach_inventory_expectations(feat, inventory, train_end):
    """
    Inventory is operational and can drift; only recent3y rates are created.
    Broad all-history inventory averages are intentionally excluded.
    """
    if inventory is None or inventory.empty or "snapshot_date" not in inventory.columns:
        return feat

    inv = inventory.copy()
    inv["snapshot_date"] = pd.to_datetime(inv["snapshot_date"], errors="coerce")
    inv = inv[(inv["snapshot_date"].notna()) & (inv["snapshot_date"] <= pd.Timestamp(train_end))].copy()
    if inv.empty:
        return feat
    inv["Date"] = inv["snapshot_date"].dt.floor("D")
    inv = _add_calendar_keys(inv, "Date")
    recent = inv[inv["Date"] >= _recent_start_for_aux(train_end, years=3)].copy()

    metrics = {
        "stockout_flag": "hist_recent3y_stockout_rate_by_monthofyear",
        "overstock_flag": "hist_recent3y_overstock_rate_by_monthofyear",
        "sell_through_rate": "hist_recent3y_sell_through_rate_by_monthofyear",
        "fill_rate": "hist_recent3y_fill_rate_by_monthofyear",
    }
    use_cols = [c for c in metrics if c in recent.columns]
    if not use_cols:
        return feat
    for c in use_cols:
        recent[c] = pd.to_numeric(recent[c], errors="coerce")
    tab = recent.groupby("monthofyear")[use_cols].mean().reset_index().rename(columns=metrics)

    out = _add_calendar_keys(feat, "Date")
    out = _merge_group_prior(out, tab, "monthofyear", [c for c in tab.columns if c != "monthofyear"])
    return out.drop(columns=["monthofyear", "quarter_key", "weekofyear_key", "dayofweek_key", "dayofmonth_key", "month_dayofweek_key", "year_key"], errors="ignore")

def attach_all_official_aux_expectations(feat, train_end, sales_df,
                                         orders=None, customers=None, payments=None,
                                         promotions=None, inventory=None, products=None):
    """
    Attach clean no-leakage features from official auxiliary datasets.
    """
    out = feat.copy()

    out = attach_orders_payment_expectations(out, orders, payments, train_end)
    out = attach_customer_lifecycle_expectations(out, orders, customers, train_end)
    out = attach_product_mix_expectations(out, inventory, products, train_end)
    out = attach_inventory_expectations(out, inventory, train_end)

    # Safe interactions between already-safe, non-weak features.
    if "hist_recent3y_expected_order_count_by_monthofyear" in out.columns and "is_last_3_days" in out.columns:
        out["is_last_3_days_x_hist_recent3y_expected_order_count_by_monthofyear"] = (
            out["is_last_3_days"] * out["hist_recent3y_expected_order_count_by_monthofyear"]
        )
    if "hist_recent3y_expected_aov_by_monthofyear" in out.columns and "is_last_3_days" in out.columns:
        out["is_last_3_days_x_hist_recent3y_expected_aov_by_monthofyear"] = (
            out["is_last_3_days"] * out["hist_recent3y_expected_aov_by_monthofyear"]
        )
    if "hist_recent3y_expected_aov_by_monthofyear" in out.columns and "flag_q4_peak" in out.columns:
        out["flag_q4_peak_x_hist_recent3y_expected_aov_by_monthofyear"] = (
            out["flag_q4_peak"] * out["hist_recent3y_expected_aov_by_monthofyear"]
        )

    # Remove helper keys that are not intended model features.
    helper_cols = ["monthofyear", "quarter_key", "weekofyear_key", "dayofweek_key", "dayofmonth_key", "month_dayofweek_key", "year_key"]
    out = out.drop(columns=[c for c in helper_cols if c in out.columns], errors="ignore")
    return out

## 4. Dynamic recursive sales features

In [14]:
# ============================================================
# 4. Dynamic lag/rolling features built from actual history or recursive predictions
# ============================================================

TARGET_LAGS = [1, 2, 3, 7, 14, 21, 28, 35, 56, 91, 182, 364, 365, 366, 371, 728, 730]
ROLL_WINDOWS = [3, 7, 14, 28, 56, 91, 182, 365]
EWM_SPANS = [7, 14, 28, 56, 91]

def _days_since_event(event_series):
    vals = event_series.fillna(0).astype(int).to_numpy()
    out = np.full(len(vals), np.nan)
    last = None
    for i, v in enumerate(vals):
        if v == 1:
            last = i
            out[i] = 0
        elif last is not None:
            out[i] = i - last
    return pd.Series(out).fillna(999).clip(0, 999)

def add_dynamic_features(df):
    """
    Add shifted target features.
    Future rows are recomputed after predictions are inserted, so row t only sees dates < t.
    """
    out = df.copy().sort_values("Date").reset_index(drop=True)

    out["_gross_profit_raw"] = out["Revenue"] - out["COGS"]
    out["_cogs_ratio_raw"] = np.where(out["Revenue"] != 0, out["COGS"] / out["Revenue"], np.nan)
    out["_gpm_raw"] = np.where(out["Revenue"] != 0, (out["Revenue"] - out["COGS"]) / out["Revenue"], np.nan)

    for target in TARGETS:
        lower = target.lower()
        shifted = out[target].shift(1)

        # Direct lags, including YoY +/- weekly offset anchors
        for lag in TARGET_LAGS:
            out[f"{lower}_lag_{lag}"] = out[target].shift(lag)

        # Rolling stats based only on past values
        for w in ROLL_WINDOWS:
            minp = max(2, min(w // 3, 28))
            out[f"{lower}_roll_mean_{w}"] = shifted.rolling(w, min_periods=minp).mean()
            out[f"{lower}_roll_std_{w}"] = shifted.rolling(w, min_periods=minp).std()
            out[f"{lower}_roll_median_{w}"] = shifted.rolling(w, min_periods=minp).median()
            out[f"{lower}_roll_min_{w}"] = shifted.rolling(w, min_periods=minp).min()
            out[f"{lower}_roll_max_{w}"] = shifted.rolling(w, min_periods=minp).max()
            out[f"{lower}_roll_q25_{w}"] = shifted.rolling(w, min_periods=minp).quantile(0.25)
            out[f"{lower}_roll_q75_{w}"] = shifted.rolling(w, min_periods=minp).quantile(0.75)
            out[f"{lower}_roll_range_{w}"] = out[f"{lower}_roll_max_{w}"] - out[f"{lower}_roll_min_{w}"]
            out[f"{lower}_roll_iqr_{w}"] = out[f"{lower}_roll_q75_{w}"] - out[f"{lower}_roll_q25_{w}"]
            out[f"{lower}_roll_cv_{w}"] = safe_divide(out[f"{lower}_roll_std_{w}"], out[f"{lower}_roll_mean_{w}"])

        for span in EWM_SPANS:
            out[f"{lower}_ewm_{span}"] = shifted.ewm(span=span, adjust=False, min_periods=3).mean()

        # Delta, percentage, acceleration. All based on past/predicted values.
        out[f"{lower}_delta_1"] = out[f"{lower}_lag_1"] - out[f"{lower}_lag_2"]
        out[f"{lower}_delta_7"] = out[f"{lower}_lag_1"] - out[f"{lower}_lag_7"]
        out[f"{lower}_pct_change_1"] = safe_divide(out[f"{lower}_delta_1"], out[f"{lower}_lag_2"])
        out[f"{lower}_pct_change_7"] = safe_divide(out[f"{lower}_delta_7"], out[f"{lower}_lag_7"])
        out[f"{lower}_acceleration"] = out[f"{lower}_delta_1"] - (out[f"{lower}_lag_2"] - out[f"{lower}_lag_3"])
        out[f"{lower}_drawdown_from_28max"] = safe_divide(out[f"{lower}_lag_1"], out[f"{lower}_roll_max_28"]) - 1
        out[f"{lower}_recovery_from_28min"] = safe_divide(out[f"{lower}_lag_1"], out[f"{lower}_roll_min_28"]) - 1

        # Spike/drop states based on previous day and rolling stats
        out[f"{lower}_prev_spike"] = (
            out[f"{lower}_lag_1"] > (out[f"{lower}_roll_median_28"] + 1.25*out[f"{lower}_roll_std_28"])
        ).astype(int)
        out[f"{lower}_prev_drop"] = (
            out[f"{lower}_delta_1"] < (-0.75*out[f"{lower}_roll_std_28"])
        ).astype(int)
        out[f"{lower}_days_since_spike"] = _days_since_event(out[f"{lower}_prev_spike"])
        out[f"{lower}_days_since_drop"] = _days_since_event(out[f"{lower}_prev_drop"])
        out[f"{lower}_post_spike_cooldown_7"] = ((out[f"{lower}_days_since_spike"] <= 7) & (out[f"{lower}_prev_spike"] == 0)).astype(int)
        out[f"{lower}_post_drop_rebound_7"] = ((out[f"{lower}_days_since_drop"] <= 7) & (out[f"{lower}_prev_drop"] == 0)).astype(int)

    # Stable YoY anchors + priors
    out["revenue_yoy_anchor"] = (
        0.35*out["revenue_lag_365"]
        + 0.20*out["revenue_lag_364"]
        + 0.20*out["revenue_lag_366"]
        + 0.15*out["revenue_lag_371"]
        + 0.10*out["revenue_lag_730"]
    )
    out["cogs_yoy_anchor"] = (
        0.35*out["cogs_lag_365"]
        + 0.20*out["cogs_lag_364"]
        + 0.20*out["cogs_lag_366"]
        + 0.15*out["cogs_lag_371"]
        + 0.10*out["cogs_lag_730"]
    )

    out["revenue_anchor_prior_blend"] = (
        0.45*out["revenue_yoy_anchor"]
        + 0.35*out["prior_revenue_blend_recent2y"]
        + 0.20*out["prior_revenue_blend"]
    )
    out["cogs_anchor_prior_blend"] = (
        0.45*out["cogs_yoy_anchor"]
        + 0.35*out["prior_cogs_blend_recent2y"]
        + 0.20*out["prior_cogs_blend"]
    )

    # Damped recursive features to reduce 18-month drift
    for target in ["revenue", "cogs"]:
        anchor = f"{target}_anchor_prior_blend"
        for base, wt in [
            ("lag_1", 0.55),
            ("lag_7", 0.50),
            ("roll_mean_7", 0.50),
            ("roll_mean_28", 0.60),
            ("ewm_14", 0.55),
        ]:
            col = f"{target}_{base}"
            if col in out.columns:
                out[f"{target}_damped_{base}_anchor"] = wt*out[col] + (1-wt)*out[anchor]

    # Drift/momentum against anchors
    out["revenue_short_momentum"] = safe_divide(out["revenue_roll_mean_7"], out["revenue_roll_mean_28"])
    out["cogs_short_momentum"] = safe_divide(out["cogs_roll_mean_7"], out["cogs_roll_mean_28"])
    out["revenue_medium_momentum"] = safe_divide(out["revenue_roll_mean_28"], out["revenue_roll_mean_91"])
    out["cogs_medium_momentum"] = safe_divide(out["cogs_roll_mean_28"], out["cogs_roll_mean_91"])

    for target in ["revenue", "cogs"]:
        out[f"{target}_vs_yoy_anchor"] = safe_divide(out[f"{target}_lag_1"], out[f"{target}_yoy_anchor"])
        out[f"{target}_vs_prior_recent"] = safe_divide(out[f"{target}_lag_1"], out[f"prior_{target}_blend_recent2y"])
        out[f"{target}_vs_anchor_prior"] = safe_divide(out[f"{target}_lag_1"], out[f"{target}_anchor_prior_blend"])
        for c in [f"{target}_vs_yoy_anchor", f"{target}_vs_prior_recent", f"{target}_vs_anchor_prior"]:
            out[f"{c}_clipped"] = pd.Series(out[c]).clip(0.20, 5.0)

    # Margin and ratio history
    shifted_gpm = out["_gpm_raw"].shift(1)
    shifted_ratio = out["_cogs_ratio_raw"].shift(1)
    shifted_gp = out["_gross_profit_raw"].shift(1)

    for lag in [1, 7, 14, 28, 91, 182, 365]:
        out[f"gpm_lag_{lag}"] = out["_gpm_raw"].shift(lag)
        out[f"cogs_ratio_lag_{lag}"] = out["_cogs_ratio_raw"].shift(lag)
        out[f"gross_profit_lag_{lag}"] = out["_gross_profit_raw"].shift(lag)

    for w in [7, 28, 91, 182, 365]:
        minp = max(3, min(w // 3, 28))
        out[f"gpm_roll_mean_{w}"] = shifted_gpm.rolling(w, min_periods=minp).mean()
        out[f"gpm_roll_std_{w}"] = shifted_gpm.rolling(w, min_periods=minp).std()
        out[f"cogs_ratio_roll_mean_{w}"] = shifted_ratio.rolling(w, min_periods=minp).mean()
        out[f"cogs_ratio_roll_std_{w}"] = shifted_ratio.rolling(w, min_periods=minp).std()
        out[f"gross_profit_roll_mean_{w}"] = shifted_gp.rolling(w, min_periods=minp).mean()

    # Ratio anchors for Experiment 2 cogs_ratio post-processing
    out["cogs_ratio_anchor_blend"] = (
        0.40*out["prior_cogs_ratio_blend_recent2y"]
        + 0.25*out["prior_cogs_ratio_blend"]
        + 0.20*out["cogs_ratio_roll_mean_365"]
        + 0.15*out["cogs_ratio_roll_mean_91"]
    )
    out["gpm_anchor_blend"] = (
        0.40*out["prior_gpm_blend_recent2y"]
        + 0.25*out["prior_gpm_blend"]
        + 0.20*out["gpm_roll_mean_365"]
        + 0.15*out["gpm_roll_mean_91"]
    )

    out["predicted_cogs_from_revenue_anchor"] = out["prior_revenue_blend_recent2y"] * out["cogs_ratio_anchor_blend"]
    out["cogs_ratio_recent_vs_history"] = out["prior_cogs_ratio_blend_recent2y"] - out["prior_cogs_ratio_blend"]

    # Interactions
    if "promo_intensity" in out.columns:
        out["prior_revenue_x_promo"] = out["prior_revenue_blend_recent2y"] * out["promo_intensity"]
        out["prior_cogs_x_promo"] = out["prior_cogs_blend_recent2y"] * out["promo_intensity"]
        out["prior_gpm_x_promo"] = out["prior_gpm_blend_recent2y"] * out["promo_intensity"]
        out["prior_ratio_x_promo"] = out["prior_cogs_ratio_blend_recent2y"] * out["promo_intensity"]
        out["post_spike_x_promo"] = out["revenue_post_spike_cooldown_7"] * out["promo_intensity"]
        out["pre_promo_x_revenue_prior"] = out.get("promo_pre_window_7", 0) * out["prior_revenue_blend_recent2y"]
    if "inv_efficiency_index" in out.columns:
        out["prior_gpm_x_inv_efficiency"] = out["prior_gpm_blend_recent2y"] * out["inv_efficiency_index"]
    if "inv_supply_tightness" in out.columns:
        out["prior_ratio_x_supply_tightness"] = out["prior_cogs_ratio_blend_recent2y"] * out["inv_supply_tightness"]
    if "traffic_engagement_score" in out.columns:
        out["traffic_x_revenue_prior"] = out["traffic_engagement_score"] * out["prior_revenue_blend_recent2y"]

    drop_raw = ["_gross_profit_raw", "_cogs_ratio_raw", "_gpm_raw"]
    return out.drop(columns=[c for c in drop_raw if c in out.columns])

## 5. Build modeling frames

In [ ]:
# ============================================================
# 5. Build modeling frame and summaries
# ============================================================

def build_modeling_frame(base_df, train_end):
    """
    Build the full leakage-safe feature table for a selected training cutoff.

    1. Static features are computed only from Date.
    2. Target priors are learned only from rows <= train_end.
    3. Official auxiliary expectations are learned only from rows <= train_end and attached by calendar keys.
    4. Dynamic target features are shifted so row t only sees dates < t.
    """
    train_end = pd.Timestamp(train_end)
    train_hist = base_df.loc[base_df["Date"] <= train_end, ["Date", "Revenue", "COGS"]].copy()

    target_priors = build_target_priors(train_hist)

    feat = add_static_features(base_df)
    feat = attach_target_priors(feat, target_priors)

    feat = attach_all_official_aux_expectations(
        feat,
        train_end=train_end,
        sales_df=base_df,
        orders=orders_df,
        customers=customers_df,
        payments=payments_df,
        promotions=promotions_df,
        inventory=inventory_df,
        products=products_df,
    )

    feat = add_dynamic_features(feat)

    # Drop cols
    col_del = ["year", "month", "day", "dayofweek", "dayofyear", "weekofyear", "quarter", 
               "is_month_end_calc", "is_quarter_start", "is_quarter_end", "days_in_month", 
               "is_first_1_day", "is_first_3_days", "is_last_1_day", "is_last_3_days",
               "is_first_week_of_month", "is_last_week_of_month", "is_mid_month",
               "is_month_start_calc"]
    feat.drop(columns=col_del, inplace=True, errors="ignore")
    # Keep clean calendar flags; HGB can decide whether they help.

    priors = {"target": target_priors}
    return feat, priors

def get_feature_columns(feat):
    blocked = {"Date", "Revenue", "COGS"}
    feature_cols = []
    for c in feat.columns:
        if c in blocked:
            continue
        if pd.api.types.is_numeric_dtype(feat[c]):
            feature_cols.append(c)
    return feature_cols


def summarize_feature_groups(columns):
    groups = []
    for c in columns:
        cl = c.lower()
        if c in {"Date", "Revenue", "COGS"}:
            g = "id_or_target"
        elif cl.startswith("hist_expected_") or cl.startswith("hist_recent3y_") or cl.startswith("hist_trend_") or cl.startswith("hist_"):
            if "order" in cl or "aov" in cl:
                g = "official_orders_historical_expectation"
            elif "customer" in cl or "reactivation" in cl or "returning" in cl or "repeat_purchase" in cl:
                g = "official_customer_lifecycle_expectation"
            elif "promo" in cl or "discount" in cl:
                g = "official_promotion_clean_features"
            elif "payment" in cl or "installment" in cl or "credit" in cl or "cod" in cl or "bank" in cl:
                g = "official_payment_share_rate_expectation"
            elif "category" in cl or "streetwear" in cl or "outdoor" in cl or "casual" in cl or "genz" in cl:
                g = "official_product_mix_expectation"
            elif "inventory" in cl or "stockout" in cl or "overstock" in cl or "sell_through" in cl or "fill_rate" in cl or "days_of_supply" in cl:
                g = "official_inventory_recent_rate_expectation"
            else:
                g = "historical_expectation_prior"
        elif c.startswith("prior_"):
            g = "sales_target_prior"
        elif "date_regime" in c:
            g = "date_regime_cluster_proxy"
        elif "damped" in c or "anchor" in c or "_vs_yoy_" in c or "_vs_prior_" in c or "_vs_anchor_" in c:
            g = "damped_anchor_or_drift_control"
        elif "spike" in c or "drop" in c or "drawdown" in c or "recovery" in c or "delta" in c or "acceleration" in c or "pct_change" in c:
            g = "spike_drop_momentum_state"
        elif "_lag_" in c or "_roll_" in c or "_ewm_" in c or c.endswith("_momentum"):
            g = "dynamic_target_history"
        elif "gpm" in c or "cogs_ratio" in c or "gross_profit" in c or "roi_proxy" in c:
            g = "margin_history_or_prior"
        elif c.startswith(("is_", "flag_", "dow_", "doy_", "month_", "week_", "sin_", "cos_")) or c in {"years_since_start", "days_to_month_end", "days_since_month_start", "month_progress", "days_to_year_end"}:
            g = "calendar"
        else:
            g = "other_numeric"
        groups.append({"feature": c, "group": g})
    return pd.DataFrame(groups)

features_backtest, priors_backtest = build_modeling_frame(df, BACKTEST_TRAIN_END)
features_final, priors_final = build_modeling_frame(df, TRAIN_END_FINAL)

feature_cols_final = get_feature_columns(features_final)
feature_cols_backtest = get_feature_columns(features_backtest)

# Guardrails: no actual-future auxiliary columns should appear.
unsafe_name_patterns = [
    "actual_2023", "actual_2024", "previous_month_", "last_month_",
    "is_promo_2023", "age_share_2023", "order_count_2023",
]
unsafe_cols = [c for c in feature_cols_final if any(p in c.lower() for p in unsafe_name_patterns)]
assert len(unsafe_cols) == 0, f"Unsafe future-like feature columns found: {unsafe_cols[:30]}"

# Clean-plan guardrails: remove weak or stale broad features.
weak_or_removed_patterns = [
    "gender", "female_share", "male_share",
    "hist_expected_payment_value", "payment_value_total", "sum_installment", "sum_installments",
    "hist_expected_inventory_pressure", "hist_expected_inventory_volatility",
    "hist_expected_promo_revenue_lift_by_monthofyear",
    "hist_expected_promo_cogs_lift_by_monthofyear",
    "hist_expected_promo_aov_lift_by_monthofyear",
    "hist_expected_promo_margin_impact_by_monthofyear",
]
weak_cols = [c for c in feature_cols_final if any(p in c.lower() for p in weak_or_removed_patterns)]
assert len(weak_cols) == 0, f"Weak/removed feature columns found: {weak_cols[:30]}"

print("Backtest feature table:", features_backtest.shape)
print("Final feature table:", features_final.shape)
print("Backtest feature columns:", len(feature_cols_backtest))
print("Final feature columns:", len(feature_cols_final))
print("Unsafe future-like feature columns found:", len(unsafe_cols))

aux_cols = [c for c in feature_cols_final if c.startswith("hist_expected_") or c.startswith("hist_recent3y_") or c.startswith("hist_trend_") or c.startswith("hist_") or c.startswith("scheduled_")]
print("Official auxiliary historical-expectation feature columns:", len(aux_cols))
display(pd.DataFrame({"aux_feature": aux_cols}).head(40))

Backtest feature table: (3833, 450)
Final feature table: (3833, 450)
Backtest feature columns: 447
Final feature columns: 447
Unsafe future-like feature columns found: 0
Official auxiliary historical-expectation feature columns: 58


,aux_feature
0,hist_expected_order_count_by_monthofyear
1,hist_expected_aov_by_monthofyear
2,hist_expected_order_count_by_weekofyear
3,hist_expected_aov_by_weekofyear
4,hist_expected_order_count_by_dayofweek
5,hist_expected_aov_by_dayofweek
6,hist_expected_order_count_by_month_dayofweek
7,hist_expected_aov_by_month_dayofweek
8,hist_recent3y_expected_order_count_by_monthofyear
9,hist_recent3y_expected_aov_by_monthofyear


## 6. Leakage and missingness checks

In [16]:
# ============================================================
# Basic checks
# ============================================================

def feature_missingness_report(feat, feature_cols, start=None, end=None):
    sub = feat.copy()
    if start is not None:
        sub = sub[sub["Date"] >= pd.Timestamp(start)]
    if end is not None:
        sub = sub[sub["Date"] <= pd.Timestamp(end)]

    report = (
        sub[feature_cols]
        .isna()
        .mean()
        .sort_values(ascending=False)
        .rename("missing_rate")
        .reset_index()
        .rename(columns={"index": "feature"})
    )
    return report

backtest_missing = feature_missingness_report(
    features_backtest,
    feature_cols_backtest,
    BACKTEST_START,
    BACKTEST_END,
)
forecast_missing = feature_missingness_report(
    features_final,
    feature_cols_final,
    FORECAST_START,
    FORECAST_END,
)

print("Top missing features in 2021–2022 validation window:")
display(backtest_missing.head(20))

print("Top missing features in 2023–2024 forecast window:")
display(forecast_missing.head(20))

assert features_backtest["Date"].is_monotonic_increasing
assert features_final["Date"].is_monotonic_increasing
assert "Revenue" not in feature_cols_final
assert "COGS" not in feature_cols_final

# For final forecast rows, auxiliary historical-expectation features should be attached without requiring actual future auxiliary data.
forecast_rows = features_final[(features_final["Date"] >= FORECAST_START) & (features_final["Date"] <= FORECAST_END)]
aux_cols = [c for c in feature_cols_final if c.startswith("hist_expected_") or c.startswith("hist_recent3y_") or c.startswith("hist_trend_") or c.startswith("hist_")]
print("Aux historical-expectation missingness in forecast rows:")
if aux_cols:
    display(forecast_rows[aux_cols].isna().mean().sort_values(ascending=False).head(20))
else:
    print("No auxiliary historical-expectation columns were created.")

Top missing features in 2021–2022 validation window:


,feature,missing_rate
0,is_weekend,0.0
1,days_to_month_end,0.0
2,days_since_month_start,0.0
3,month_progress,0.0
4,week_of_month,0.0
5,days_to_year_end,0.0
6,years_since_start,0.0
7,sin_y1,0.0
8,cos_y1,0.0
9,sin_y2,0.0


Top missing features in 2023–2024 forecast window:


,feature,missing_rate
0,is_weekend,NaN
1,days_to_month_end,NaN
2,days_since_month_start,NaN
3,month_progress,NaN
4,week_of_month,NaN
5,days_to_year_end,NaN
6,years_since_start,NaN
7,sin_y1,NaN
8,cos_y1,NaN
9,sin_y2,NaN


Aux historical-expectation missingness in forecast rows:


hist_expected_order_count_by_monthofyear              NaN
hist_expected_aov_by_monthofyear                      NaN
hist_expected_order_count_by_weekofyear               NaN
hist_expected_aov_by_weekofyear                       NaN
hist_expected_order_count_by_dayofweek                NaN
hist_expected_aov_by_dayofweek                        NaN
hist_expected_order_count_by_month_dayofweek          NaN
hist_expected_aov_by_month_dayofweek                  NaN
hist_recent3y_expected_order_count_by_monthofyear     NaN
hist_recent3y_expected_aov_by_monthofyear             NaN
hist_order_count_recent_vs_all_ratio_by_monthofyear   NaN
hist_aov_recent_vs_all_ratio_by_monthofyear           NaN
hist_order_count_trend_slope_by_monthofyear           NaN
hist_aov_trend_slope_by_monthofyear                   NaN
hist_recent3y_delivered_rate_by_monthofyear           NaN
hist_recent3y_returned_rate_by_monthofyear            NaN
hist_recent3y_cancelled_rate_by_monthofyear           NaN
hist_recent3y_

## 7. Save outputs

In [18]:
# ============================================================
# Save outputs
# ============================================================
# os.chdir("./Feature Engineering")

OUTPUT_DIR = "./"

backtest_out = OUTPUT_DIR + "features_validate.csv"
final_out = OUTPUT_DIR + "final_features_forecast.csv"
feature_cols_out = OUTPUT_DIR + "feature_columns_forecast.json"
groups_out = OUTPUT_DIR + "feature_groups_forecast.csv"
summary_out = OUTPUT_DIR + "feature_engineering_summary.csv"

features_backtest.to_csv(backtest_out, index=False)
features_final.to_csv(final_out, index=False)

with open(feature_cols_out, "w") as f:
    json.dump(feature_cols_final, f, indent=2)

feature_group_summary = summarize_feature_groups(feature_cols_final)
summary = (
    feature_group_summary
    .groupby("group")
    .size()
    .sort_values(ascending=False)
    .rename("n_features")
    .reset_index()
)
feature_group_summary.to_csv(groups_out, index=False)
summary.to_csv(summary_out, index=False)

print("Saved:")
print(backtest_out)
print(final_out)
print(feature_cols_out)
print(groups_out)
print(summary_out)

display(summary)

Saved:
./features_validate.csv
./final_features_forecast.csv
./feature_columns_forecast.json
./feature_groups_forecast.csv
./feature_engineering_summary.csv


,group,n_features
0,dynamic_target_history,254
1,sales_target_prior,38
2,calendar,34
3,damped_anchor_or_drift_control,29
4,spike_drop_momentum_state,26
5,official_product_mix_expectation,20
6,official_orders_historical_expectation,14
7,official_customer_lifecycle_expectation,8
8,official_payment_share_rate_expectation,8
9,date_regime_cluster_proxy,4


In [ ]:
!cd

c:\Users\Quoc Thai\Downloads\Datathon\Feature Engineering
